# CommaSeparatedListOutputParser → 구조화 출력의 `list[str]`

`CommaSeparatedListOutputParser`는 `langchain_core`에 그대로 남아 있고 폐기(deprecated)되지 않았습니다. 가볍게 쓰기에는 여전히 괜찮습니다.

다만 **항목 안에 쉼표가 들어가는 경우**(예: `"서울, 경복궁"`) 목록이 잘못 쪼개지는 근본적 한계가 있습니다. 현재 권장 방식은 `list[str]` 필드를 가진 스키마로 **구조화 출력**을 받는 것입니다.

| 구분 | 책의 방식 | 현재 권장 방식 |
|---|---|---|
| 프롬프트 | `PromptTemplate(..., partial_variables=...)` | `ChatPromptTemplate` + `.partial()` |
| 모델 | `ChatOpenAI(temperature=0)` (모델명 생략) | `init_chat_model("openai:...")` — 모델명을 명시 |
| 목록 파싱 | 문자열을 쉼표로 split | `with_structured_output()` 로 JSON 배열 수신 |

In [ ]:
# 최초 1회 설치 (LangChain v1 기준)
# %pip install -qU langchain langchain-openai langchain-classic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 의 OPENAI_API_KEY, LANGSMITH_API_KEY 를 불러옵니다.

# LangSmith 추적: 별도 헬퍼 없이 환경변수만 설정하면 자동으로 활성화됩니다.
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "CH03-OutputParser")

In [ ]:
from langchain.chat_models import init_chat_model

# 공급자 중립적인 모델 초기화 ("공급자:모델명")
# 다른 모델로 바꾸려면 문자열만 교체하면 됩니다. 예) "anthropic:claude-sonnet-4-5", "ollama:llama3.1"
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

## 1. `CommaSeparatedListOutputParser` (최신 문법으로 정리)

`PromptTemplate(template=..., input_variables=..., partial_variables=...)` 대신, 채팅 모델에 맞는 `ChatPromptTemplate`을 쓰고 `.partial()`로 지침을 채웁니다. `input_variables`는 템플릿에서 자동 추론되므로 적지 않아도 됩니다.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import ChatPromptTemplate

output_parser = CommaSeparatedListOutputParser()

prompt = ChatPromptTemplate.from_template(
    "List five {subject}.\n{format_instructions}"
).partial(format_instructions=output_parser.get_format_instructions())

chain = prompt | llm | output_parser

chain.invoke({"subject": "대한민국 관광명소"})

이 파서는 스트리밍 시 항목이 완성될 때마다 `['항목']` 형태로 하나씩 내보냅니다.

In [ ]:
for s in chain.stream({"subject": "대한민국 관광명소"}):
    print(s)

## 2. 현재 권장 방식: `list[str]` 스키마 + `with_structured_output()`

형식 지침이 필요 없고, 항목에 쉼표가 들어가도 안전합니다. 마지막 단계의 `lambda`는 LCEL에서 자동으로 `RunnableLambda`로 변환되어 `list[str]`만 꺼내 줍니다.

In [ ]:
from pydantic import BaseModel, Field


class ItemList(BaseModel):
    """요청한 주제에 해당하는 항목 목록"""

    items: list[str] = Field(description="항목 이름 목록")


list_prompt = ChatPromptTemplate.from_template("{subject} {n}개를 나열하세요.")

list_chain = list_prompt | llm.with_structured_output(ItemList) | (lambda r: r.items)

list_chain.invoke({"subject": "대한민국 관광명소", "n": 5})

쉼표가 포함된 항목도 올바르게 유지되는지 확인해 봅니다.

In [ ]:
list_chain.invoke({"subject": "'도시, 관광명소' 형식으로 표기한 대한민국 관광명소", "n": 3})

여러 입력을 한 번에 처리할 때는 `batch()`를 사용합니다(내부적으로 병렬 실행).

In [ ]:
list_chain.batch(
    [
        {"subject": "대한민국 관광명소", "n": 3},
        {"subject": "한국 전통 음식", "n": 3},
    ]
)